In [ ]:
# In[1]:

import os
import pandas as pd
import numpy as np

# helper to load csv if present, else empty DataFrame with expected columns
def load_csv_if_exists(fname, expected_cols):
    if os.path.exists(fname):
        df = pd.read_csv(fname)
        # Ensure expected columns exist; if not, keep what we have
        for c in expected_cols:
            if c not in df.columns:
                df[c] = pd.NA
        return df
    else:
        # empty df with expected columns
        return pd.DataFrame(columns=expected_cols)

# Load files (error_logs.csv optional)
metric = load_csv_if_exists("metric.csv", ["timestamp", "cmdb_id", "kpi_name", "value"])
trace = load_csv_if_exists("trace.csv", ["timestamp", "cmdb_id", "trace_name", "value"])
log = load_csv_if_exists("log.csv", ["timestamp", "cmdb_id", "log_name", "value"])
error_logs = load_csv_if_exists("error_logs.csv", ["timestamp", "cmdb_id", "message"])

# Parse timestamp as datetime (unit='s', utc=True) when column exists and non-empty
for df in (metric, trace, log, error_logs):
    if "timestamp" in df.columns and not df["timestamp"].empty:
        # if column may be non-numeric, coerce errors
        df["timestamp"] = pd.to_datetime(pd.to_numeric(df["timestamp"], errors="coerce"), unit="s", utc=True)
    else:
        # ensure column exists with correct dtype
        df["timestamp"] = pd.to_datetime([], unit="s", utc=True)

# Aggregation function
def aggregate_telemetry(df, group_cols, value_col="value", name_label=None):
    # If dataframe empty or no numeric values, return empty structured df
    if df.shape[0] == 0 or value_col not in df.columns:
        cols = group_cols + ["count", "ts_min", "ts_max", "mean_value", "max_value", "p95_value"]
        return pd.DataFrame(columns=cols)
    # Ensure numeric value column
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    grp = df.groupby(group_cols, dropna=False)
    agg = grp.agg(
        count=(value_col, "count"),
        ts_min=("timestamp", "min"),
        ts_max=("timestamp", "max"),
        mean_value=(value_col, "mean"),
        max_value=(value_col, "max"),
    ).reset_index()
    # compute p95 separately (to avoid issues with named aggregations)
    p95 = grp[value_col].quantile(0.95).reset_index().rename(columns={value_col: "p95_value"})
    agg = agg.merge(p95, on=group_cols, how="left")
    # sort by count desc, keep top 20
    agg = agg.sort_values("count", ascending=False).reset_index(drop=True)
    return agg.head(20)

# Perform aggregations
metric_agg = aggregate_telemetry(metric, ["cmdb_id", "kpi_name"], value_col="value")
trace_agg = aggregate_telemetry(trace, ["cmdb_id", "trace_name"], value_col="value")
log_agg = aggregate_telemetry(log, ["cmdb_id", "log_name"], value_col="value")

# Distinct cmdb_id values across all files
cmdbs_metric = set(metric["cmdb_id"].dropna().unique()) if "cmdb_id" in metric.columns else set()
cmdbs_trace = set(trace["cmdb_id"].dropna().unique()) if "cmdb_id" in trace.columns else set()
cmdbs_log = set(log["cmdb_id"].dropna().unique()) if "cmdb_id" in log.columns else set()
cmdbs_error = set(error_logs["cmdb_id"].dropna().unique()) if "cmdb_id" in error_logs.columns else set()

distinct_cmdbs = sorted(set().union(cmdbs_metric, cmdbs_trace, cmdbs_log, cmdbs_error))

# Candidate services presence
candidates = ["checkoutservice", "currencyservice", "emailservice", "productcatalogservice", "recommendationservice"]
candidates_present = sorted([c for c in candidates if c in distinct_cmdbs])

# Compact outputs: aggregated tables and distinct cmdb list + found candidates
metric_agg, trace_agg, log_agg, distinct_cmdbs, candidates_present

```
Out[1]:
```
summary = (
    "Summary of telemetry scan:\n"
    "- Distinct services observed (13): adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, frontend-external, frontendservice, paymentservice, productcatalogservice, recommendationservice, redis, shippingservice.\n"
    "- Candidate services present: checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice.\n"
    "- Metric highlights: checkoutservice shows elevated request latencies (latency-50 mean ~0.099s, latency-90 mean ~0.233s, P95 ~0.32s). "
    "Adservice has very large disk I/O and high memory (diskio P95 ~8.9e4, mem P95 ~1.05e8).\n"
    "- Trace highlights: many checkoutservice traces present; frontend→checkoutservice P95s are elevated (trace p95 up to ~0.31–0.86s in the series), but trace error_rate fields are 0.0.\n"
    "- Log highlights: log.error_count is 0 for observed services (no log error spikes). High request volume (log.row_count) is seen for frontend, currencyservice and recommendationservice.\n"
    "- Overall conclusion: the clearest anomalous signal is elevated latency centered on checkoutservice (seen in both metric and trace telemetry). There are no corresponding log error spikes; adservice shows heavy disk I/O and memory usage that may be relevant for resource contention but checkoutservice latency is the most prominent issue."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id    kpi_name  count                    ts_min                    ts_max    mean_value     max_value     p95_value
0         adservice         cpu     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  1.779202e+00  4.444609e+00  3.684557e+00
1         adservice      diskio     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  6.616206e+04  1.321266e+05  8.943692e+04
2         adservice  latency-50     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  3.053399e-03  3.642855e-03  3.105631e-03
3         adservice  latency-90     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  4.765471e-03  7.458398e-03  4.790136e-03
4         adservice         mem     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  1.030974e+08  1.064630e+08  1.052749e+08
5         adservice      socket     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  4.000000e+00  4.000000e+00  4.000000e+00
6         adservice    workload     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  1.485802e+01  1.616306e+01  1.566893e+01
7       cartservice         cpu     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  2.290638e+00  2.865066e+00  2.812663e+00
8       cartservice      diskio     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  0.000000e+00  0.000000e+00  0.000000e+00
9       cartservice  latency-50     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  3.123587e-03  3.265597e-03  3.256283e-03
10      cartservice  latency-90     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  4.933041e-03  6.099141e-03  5.774443e-03
11      cartservice         mem     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  4.309915e+07  4.444890e+07  4.437462e+07
12      cartservice      socket     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  4.000000e+00  4.000000e+00  4.000000e+00
13      cartservice    workload     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  2.506036e+01  2.728530e+01  2.670272e+01
14  checkoutservice         cpu     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  4.174765e-01  5.338228e-01  4.802671e-01
15  checkoutservice      diskio     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  0.000000e+00  0.000000e+00  0.000000e+00
16  checkoutservice  latency-50     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  9.898495e-02  1.714903e-01  1.649606e-01
17  checkoutservice  latency-90     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  2.334883e-01  4.835432e-01  3.190125e-01
18  checkoutservice         mem     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  1.024974e+07  1.088887e+07  1.078679e+07
19  checkoutservice      socket     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  9.000000e+00  9.000000e+00  9.000000e+00,             cmdb_id                                trace_name  count                    ts_min                    ts_max  mean_value   max_value   p95_value
0   checkoutservice  trace.from_checkoutservice.duration_mean     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00    0.006307    0.026346    0.016872
1   checkoutservice   trace.from_checkoutservice.duration_p95     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00    0.018545    0.090550    0.081494
2   checkoutservice     trace.from_checkoutservice.error_rate     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00    0.000000    0.000000    0.000000
3   checkoutservice      trace.from_checkoutservice.row_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  222.040000  287.000000  284.200000
4   checkoutservice  trace.from_frontendservice.duration_mean     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00    0.101655    0.233504    0.173561
5   checkoutservice   trace.from_frontendservice.duration_p95     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00    0.203552    0.860552    0.310855
6   checkoutservice     trace.from_frontendservice.error_rate     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00    0.000000    0.000000    0.000000
7   checkoutservice      trace.from_frontendservice.row_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00   26.600000   34.000000   33.800000
8   checkoutservice             trace.from_root.duration_mean     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00    0.011627    0.015851    0.015282
9   checkoutservice              trace.from_root.duration_p95     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00    0.059220    0.074843    0.074224
10  checkoutservice                trace.from_root.error_rate     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00    0.000000    0.000000    0.000000
11  checkoutservice                 trace.from_root.row_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  112.080000  144.000000  139.800000
12  checkoutservice    trace.to_checkoutservice.duration_mean     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00    0.006307    0.026346    0.016872
13  checkoutservice     trace.to_checkoutservice.duration_p95     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00    0.018545    0.090550    0.081494
14  checkoutservice       trace.to_checkoutservice.error_rate     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00    0.000000    0.000000    0.000000
15  checkoutservice        trace.to_checkoutservice.row_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  222.040000  287.000000  284.200000
16  checkoutservice    trace.to_currencyservice.duration_mean     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00    0.000758    0.002244    0.001725
17  checkoutservice     trace.to_currencyservice.duration_p95     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00    0.000222    0.000282    0.000257
18  checkoutservice       trace.to_currencyservice.error_rate     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00    0.000000    0.000000    0.000000
19  checkoutservice        trace.to_currencyservice.row_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00   89.040000  120.000000  115.800000,                   cmdb_id         log_name  count                    ts_min                    ts_max  mean_value  max_value  p95_value
0               adservice  log.error_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00        0.00          0        0.0
1               adservice    log.row_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00      426.84        472      462.8
2             cartservice  log.error_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00        0.00          0        0.0
3             cartservice    log.row_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00      731.36        816      803.4
4         checkoutservice  log.error_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00        0.00          0        0.0
5         checkoutservice    log.row_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00       79.80        102      101.4
6         currencyservice  log.error_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00        0.00          0        0.0
7         currencyservice    log.row_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00     2262.56       2561     2553.8
8            emailservice  log.error_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00        0.00          0        0.0
9            emailservice    log.row_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00       26.60         34       33.8
10               frontend  log.error_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00        0.00          0        0.0
11               frontend    log.row_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00     2539.48       2860     2811.8
12         paymentservice  log.error_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00        0.00          0        0.0
13         paymentservice    log.row_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00       53.20         68       67.6
14  recommendationservice  log.error_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00        0.00          0        0.0
15  recommendationservice    log.row_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00      532.04        585      583.2
16        shippingservice    log.row_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00      428.16        494      485.2
17        shippingservice  log.error_count     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00        0.00          0        0.0
18                  redis    log.row_count      5 2024-01-17 16:32:00+00:00 2024-01-17 16:52:00+00:00        5.00          5        5.0
19                  redis  log.error_count      5 2024-01-17 16:32:00+00:00 2024-01-17 16:52:00+00:00        0.00          0        0.0, ['adservice', 'cartservice', 'checkoutservice', 'currencyservice', 'emailservice', 'frontend', 'frontend-external', 'frontendservice', 'paymentservice', 'productcatalogservice', 'recommendationservice', 'redis', 'root', 'shippingservice'], ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice'])```
```

In [ ]:
# In[2]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[2]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[3]:

import os
import pandas as pd
import numpy as np

# Load metric.csv (fresh)
metric = pd.read_csv("metric.csv")

# Parse timestamp safely: coerce non-numeric, and drop values outside a sensible unix seconds range before to_datetime
ts_num = pd.to_numeric(metric.get("timestamp"), errors="coerce")

# Define reasonable bounds (years ~2000 to ~2035)
min_ts, max_ts = 946684800, 2082758400  # 2000-01-01 to 2036-01-01
ts_num_safe = ts_num.where((ts_num >= min_ts) & (ts_num <= max_ts))

metric["timestamp"] = pd.to_datetime(ts_num_safe, unit="s", utc=True)

# Ensure numeric value column
metric["value"] = pd.to_numeric(metric.get("value", pd.Series(dtype=float)), errors="coerce")

# Candidate services to restrict to
candidates = ["checkoutservice", "currencyservice", "emailservice", "productcatalogservice", "recommendationservice"]

# Filter to candidate services
metric_candidates = metric[metric["cmdb_id"].isin(candidates)].copy()

# Aggregation for candidates
cols = ["cmdb_id", "kpi_name", "count", "ts_min", "ts_max", "mean_value", "max_value", "p95_value"]
if metric_candidates.shape[0] == 0:
    candidate_metric_agg = pd.DataFrame(columns=cols)
else:
    grp = metric_candidates.groupby(["cmdb_id", "kpi_name"], dropna=False)
    agg = grp.agg(
        count=("value", "count"),
        ts_min=("timestamp", "min"),
        ts_max=("timestamp", "max"),
        mean_value=("value", "mean"),
        max_value=("value", "max"),
    ).reset_index()
    p95 = grp["value"].quantile(0.95).reset_index().rename(columns={"value": "p95_value"})
    candidate_metric_agg = agg.merge(p95, on=["cmdb_id", "kpi_name"], how="left")
    candidate_metric_agg = candidate_metric_agg.sort_values("p95_value", ascending=False).reset_index(drop=True).head(50)

# Display compact aggregated rows
candidate_metric_agg.iloc[:, :]

```
Out[3]:
```
Summary of metric aggregates for the candidate services (using full metric.csv, sorted by global P95):

- Overall: 33 candidate (cmdb_id, kpi_name) groups were found (each group has 25 points covering 2024-01-17 16:29:00+00:00 through 2024-01-17 16:53:00+00:00 in this dataset).
- Top groups by global P95 are memory KPIs:
  - recommendationservice — mem: P95 ≈ 4.5858e7 (mean ≈ 4.5218e7, max ≈ 4.5877e7)
  - currencyservice — mem: P95 ≈ 4.5521e7 (mean ≈ 4.3983e7)
  - emailservice — mem: P95 ≈ 4.3194e7 (mean ≈ 4.3145e7)
  - productcatalogservice — mem: P95 ≈ 2.0380e7 (mean ≈ 1.3643e7)
  - checkoutservice — mem: P95 ≈ 1.0787e7 (mean ≈ 1.0250e7)
- Many latency and trace-related KPIs are present too; their P95 values are much smaller (milliseconds) and appear lower in the sorted list than memory KPIs.
- Several diskio series for these services are all zeros (P95 = 0).
- All groups in the output have count = 25 and identical ts_min/ts_max (the same 25-minute window present in the file).
- Takeaway: memory usage KPIs dominate the global P95 ranking for the candidate services, with recommendationservice and currencyservice showing the highest memory P95 values.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count                    ts_min                    ts_max    mean_value     max_value     p95_value
0   recommendationservice         mem     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  4.521789e+07  4.587679e+07  4.585775e+07
1         currencyservice         mem     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  4.398334e+07  4.570440e+07  4.552096e+07
2            emailservice         mem     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  4.314527e+07  4.319596e+07  4.319416e+07
3   productcatalogservice         mem     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  1.364309e+07  2.072180e+07  2.037962e+07
4         checkoutservice         mem     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  1.024974e+07  1.088887e+07  1.078679e+07
..                    ...         ...    ...                       ...                       ...           ...           ...           ...
28        currencyservice  latency-50     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  3.623459e-03  4.026521e-03  3.904246e-03
29           emailservice  latency-50     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  3.017197e-03  3.065539e-03  3.057535e-03
30        checkoutservice      diskio     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  0.000000e+00  0.000000e+00  0.000000e+00
31           emailservice      diskio     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  0.000000e+00  0.000000e+00  0.000000e+00
32  recommendationservice      diskio     25 2024-01-17 16:29:00+00:00 2024-01-17 16:53:00+00:00  0.000000e+00  0.000000e+00  0.000000e+00

[33 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[4]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Load metric.csv and parse timestamp/value
metric = pd.read_csv("metric.csv")
metric["timestamp"] = pd.to_datetime(pd.to_numeric(metric["timestamp"], errors="coerce"), unit="s", utc=True)
metric["value"] = pd.to_numeric(metric["value"], errors="coerce")

# Restrict to checkoutservice
m_ch = metric[metric["cmdb_id"] == "checkoutservice"].copy()

# If no data, return empty DataFrame with required columns
out_cols = [
    "cmdb_id",
    "kpi_name",
    "global_p95",
    "incident_point_count",
    "breach_count",
    "earliest_breach_timestamp",
    "max_value_in_incident",
    "severity",
]
if m_ch.empty:
    result = pd.DataFrame(columns=out_cols)
else:
    # 1) Compute global P95 per (checkoutservice, kpi_name) using full series
    global_p95 = m_ch.groupby("kpi_name", dropna=False)["value"].quantile(0.95).reset_index().rename(columns={"value": "global_p95"})

    # 2) Filter to incident window (inclusive)
    start = pd.Timestamp("2024-01-17 16:26:17", tz="UTC")
    end = pd.Timestamp("2024-01-17 16:56:17", tz="UTC")
    m_win = m_ch[(m_ch["timestamp"] >= start) & (m_ch["timestamp"] <= end)].copy()

    if m_win.empty:
        result = pd.DataFrame(columns=out_cols)
    else:
        # Merge global_p95 into incident-window rows
        m_win = m_win.merge(global_p95, on="kpi_name", how="left")

        # Mark breaches
        m_win["is_breach"] = m_win["value"] > m_win["global_p95"]

        # Aggregates in window
        agg_win = m_win.groupby("kpi_name", dropna=False).agg(
            incident_point_count=("value", "count"),
            breach_count=("is_breach", "sum"),
            max_value_in_incident=("value", "max"),
        ).reset_index()

        # Earliest breach timestamp per kpi_name
        breaches = m_win[m_win["is_breach"]].copy()
        if breaches.empty:
            earliest = pd.DataFrame(columns=["kpi_name", "earliest_breach_timestamp"])
        else:
            earliest = breaches.groupby("kpi_name", dropna=False)["timestamp"].min().reset_index().rename(columns={"timestamp": "earliest_breach_timestamp"})

        # Combine
        summary = agg_win.merge(global_p95, on="kpi_name", how="left").merge(earliest, on="kpi_name", how="left")

        # Compute severity (handle global_p95 == 0)
        summary["severity"] = np.where(summary["global_p95"] == 0,
                                       np.inf,
                                       summary["max_value_in_incident"] / summary["global_p95"])

        # Add cmdb_id column (checkoutservice)
        summary["cmdb_id"] = "checkoutservice"

        # Filter rows where breach_count > 0
        result = summary[summary["breach_count"] > 0].copy()

        # Select and order columns as requested
        result = result[["cmdb_id", "kpi_name", "global_p95", "incident_point_count", "breach_count", "earliest_breach_timestamp", "max_value_in_incident", "severity"]]

        # Sort by severity desc and limit to top 20
        result = result.sort_values(["severity", "max_value_in_incident"], ascending=[False, False]).head(20).reset_index(drop=True)

# Display compact result
result

```
Out[5]:
```
summary = (
    "Checkoutservice summary for the incident window (2024-01-17 16:26:17 → 16:56:17 UTC):\n"
    "- Multiple checkoutservice KPIs breached their global P95 during the window. Five KPI series had breaches: latency-90, cpu, latency-50, workload, and mem.\n"
    "- The most severe breach was latency-90 (max 0.4835 s vs global P95 0.3190 s, severity ≈ 1.52x); earliest breach at 2024-01-17 16:42:00 UTC.\n"
    "- CPU and latency-50 also breached (CPU severity ≈ 1.11x, earliest CPU breach 16:46; latency-50 severity ≈ 1.04x, earliest 16:42).\n"
    "- Workload and memory slightly exceeded their global P95 (severity ≈ 1.03x and 1.01x, earliest memory breach at 16:50).\n"
    "- All breached series had 2 breach points in the window and 25 samples total. Conclusion: elevated request latency (latency-90) is the clearest, highest-severity signal for checkoutservice; CPU and workload increases may be contributing, while memory shows only a marginal exceedance."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name    global_p95  incident_point_count  breach_count earliest_breach_timestamp  max_value_in_incident  severity
0  checkoutservice  latency-90  3.190125e-01                    25             2 2024-01-17 16:42:00+00:00           4.835432e-01  1.515750
1  checkoutservice         cpu  4.802671e-01                    25             2 2024-01-17 16:46:00+00:00           5.338228e-01  1.111512
2  checkoutservice  latency-50  1.649606e-01                    25             2 2024-01-17 16:42:00+00:00           1.714903e-01  1.039583
3  checkoutservice    workload  1.096735e+00                    25             2 2024-01-17 16:36:00+00:00           1.134300e+00  1.034252
4  checkoutservice         mem  1.078679e+07                    25             2 2024-01-17 16:50:00+00:00           1.088887e+07  1.009464```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Load and parse trace.csv
trace = pd.read_csv("trace.csv")
trace["timestamp"] = pd.to_datetime(pd.to_numeric(trace.get("timestamp"), errors="coerce"), unit="s", utc=True)
trace["value"] = pd.to_numeric(trace.get("value"), errors="coerce")

# Restrict to checkoutservice
t_ch = trace[trace["cmdb_id"] == "checkoutservice"].copy()

# If no data for checkoutservice, return empty DataFrame with requested columns
out_cols = [
    "cmdb_id",
    "trace_name",
    "global_p95",
    "incident_point_count",
    "breach_count",
    "earliest_breach_timestamp",
    "max_value_in_incident",
    "severity",
]
if t_ch.empty:
    trace_breaches = pd.DataFrame(columns=out_cols)
else:
    # 1) Compute global P95 per (checkoutservice, trace_name) using full series (no time filtering)
    global_p95 = t_ch.groupby("trace_name", dropna=False)["value"].quantile(0.95).reset_index().rename(columns={"value": "global_p95"})

    # 2) Filter to incident window (inclusive)
    start = pd.Timestamp("2024-01-17 16:26:17", tz="UTC")
    end = pd.Timestamp("2024-01-17 16:56:17", tz="UTC")
    t_win = t_ch[(t_ch["timestamp"] >= start) & (t_ch["timestamp"] <= end)].copy()

    if t_win.empty:
        trace_breaches = pd.DataFrame(columns=out_cols)
    else:
        # Merge global_p95 into incident-window rows
        t_win = t_win.merge(global_p95, on="trace_name", how="left")

        # Mark breaches where value > global_p95
        t_win["is_breach"] = t_win["value"] > t_win["global_p95"]

        # Aggregates in window per trace_name
        agg_win = t_win.groupby("trace_name", dropna=False).agg(
            incident_point_count=("value", "count"),
            breach_count=("is_breach", "sum"),
            max_value_in_incident=("value", "max"),
        ).reset_index()

        # Earliest breach timestamp per trace_name
        breaches = t_win[t_win["is_breach"]].copy()
        if breaches.empty:
            earliest = pd.DataFrame(columns=["trace_name", "earliest_breach_timestamp"])
        else:
            earliest = breaches.groupby("trace_name", dropna=False)["timestamp"].min().reset_index().rename(columns={"timestamp": "earliest_breach_timestamp"})

        # Combine aggregates with global_p95 and earliest breach
        summary = agg_win.merge(global_p95, on="trace_name", how="left").merge(earliest, on="trace_name", how="left")

        # Compute severity, handle global_p95 == 0
        summary["severity"] = np.where(summary["global_p95"] == 0,
                                       np.inf,
                                       summary["max_value_in_incident"] / summary["global_p95"])

        # Add cmdb_id column
        summary["cmdb_id"] = "checkoutservice"

        # Filter rows where breach_count > 0
        trace_breaches = summary[summary["breach_count"] > 0].copy()

        # Select and order columns as requested
        trace_breaches = trace_breaches[["cmdb_id", "trace_name", "global_p95", "incident_point_count", "breach_count", "earliest_breach_timestamp", "max_value_in_incident", "severity"]]

        # Sort by severity desc, then max_value_in_incident desc, limit to top 20
        trace_breaches = trace_breaches.sort_values(["severity", "max_value_in_incident"], ascending=[False, False]).head(20).reset_index(drop=True)

# Display compact result
trace_breaches

```
Out[6]:
```
summary = (
    "Trace summary for checkoutservice (window 2024-01-17 16:26:17 → 16:56:17 UTC):\n"
    "- Multiple checkoutservice trace metrics breached their global P95 during the incident window.\n"
    "- Highest-severity breach: trace.from_frontendservice.duration_p95 (max 0.8606 s vs global P95 0.3109 s, severity ≈ 2.77×); earliest breach at 2024-01-17 16:41:00 UTC.\n"
    "- Other strong breaches: to_productcatalogservice.duration_mean (≈1.65×, earliest 16:35), from/to_checkoutservice.duration_mean (≈1.56×, earliest 16:41), to_paymentservice.duration_p95 (≈1.36×, earliest 16:29), and from_frontendservice.duration_mean (≈1.35×, earliest 16:41).\n"
    "- Many additional checkoutservice traces (calls to currency, email, productcatalog, payment) show milder exceedances (severity ~1.03–1.25). Each breached series had 2 breach points in the 25-sample window.\n"
    "- Temporal pattern: the largest latency spike is concentrated around ~16:41 UTC, indicating a short-duration performance degradation affecting frontend→checkoutservice and internal checkoutservice durations.\n"
    "- Interpretation: evidence points to a latency/performance slowdown of checkoutservice during the incident window (not an error-rate spike). Investigate checkoutservice runtime resources and downstream dependencies around 16:41 UTC."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                    trace_name  global_p95  incident_point_count  breach_count earliest_breach_timestamp  max_value_in_incident  severity
0   checkoutservice       trace.from_frontendservice.duration_p95    0.310855                    25             2 2024-01-17 16:41:00+00:00               0.860552  2.768338
1   checkoutservice  trace.to_productcatalogservice.duration_mean    0.000019                    25             2 2024-01-17 16:35:00+00:00               0.000032  1.651121
2   checkoutservice      trace.from_checkoutservice.duration_mean    0.016872                    25             2 2024-01-17 16:41:00+00:00               0.026346  1.561560
3   checkoutservice        trace.to_checkoutservice.duration_mean    0.016872                    25             2 2024-01-17 16:41:00+00:00               0.026346  1.561560
4   checkoutservice          trace.to_paymentservice.duration_p95    0.001337                    25             2 2024-01-17 16:29:00+00:00               0.001820  1.361216
5   checkoutservice      trace.from_frontendservice.duration_mean    0.173561                    25             2 2024-01-17 16:41:00+00:00               0.233504  1.345373
6   checkoutservice        trace.to_currencyservice.duration_mean    0.001725                    25             2 2024-01-17 16:31:00+00:00               0.002244  1.301079
7   checkoutservice         trace.to_paymentservice.duration_mean    0.000574                    25             2 2024-01-17 16:29:00+00:00               0.000731  1.272948
8   checkoutservice   trace.to_productcatalogservice.duration_p95    0.000036                    25             2 2024-01-17 16:29:00+00:00               0.000045  1.249310
9   checkoutservice            trace.to_emailservice.duration_p95    0.000550                    25             2 2024-01-17 16:41:00+00:00               0.000684  1.243090
10  checkoutservice       trace.from_checkoutservice.duration_p95    0.081494                    25             2 2024-01-17 16:41:00+00:00               0.090550  1.111124
11  checkoutservice         trace.to_checkoutservice.duration_p95    0.081494                    25             2 2024-01-17 16:41:00+00:00               0.090550  1.111124
12  checkoutservice         trace.to_currencyservice.duration_p95    0.000257                    25             2 2024-01-17 16:30:00+00:00               0.000282  1.095997
13  checkoutservice           trace.to_emailservice.duration_mean    0.000338                    25             2 2024-01-17 16:39:00+00:00               0.000360  1.064891
14  checkoutservice      trace.to_productcatalogservice.row_count   81.800000                    25             2 2024-01-17 16:36:00+00:00              87.000000  1.063570
15  checkoutservice                 trace.from_root.duration_mean    0.015282                    25             2 2024-01-17 16:30:00+00:00               0.015851  1.037179
16  checkoutservice            trace.to_currencyservice.row_count  115.800000                    25             2 2024-01-17 16:36:00+00:00             120.000000  1.036269
17  checkoutservice                     trace.from_root.row_count  139.800000                    25             2 2024-01-17 16:36:00+00:00             144.000000  1.030043
18  checkoutservice          trace.from_checkoutservice.row_count  284.200000                    25             2 2024-01-17 16:36:00+00:00             287.000000  1.009852
19  checkoutservice            trace.to_checkoutservice.row_count  284.200000                    25             2 2024-01-17 16:36:00+00:00             287.000000  1.009852```
```

In [ ]:
# In[7]:

import os
import pandas as pd
import numpy as np

# Incident window (UTC)
start = pd.Timestamp("2024-01-17 16:26:17", tz="UTC")
end = pd.Timestamp("2024-01-17 16:56:17", tz="UTC")

candidates = ["checkoutservice", "currencyservice", "emailservice", "productcatalogservice", "recommendationservice"]

# Load log.csv (ensure fresh parsing)
log = pd.read_csv("log.csv")
log["timestamp"] = pd.to_datetime(pd.to_numeric(log.get("timestamp"), errors="coerce"), unit="s", utc=True)
log["value"] = pd.to_numeric(log.get("value"), errors="coerce")

# Filter to incident window
log_win = log[(log["timestamp"] >= start) & (log["timestamp"] <= end)].copy()

# Restrict to candidate services
log_win_cand = log_win[log_win["cmdb_id"].isin(candidates)].copy()

# If empty, prepare empty summary
if log_win_cand.empty:
    log_summary = pd.DataFrame(columns=["cmdb_id", "log_name", "incident_point_count", "mean_value", "max_value", "count_nonzero"])
else:
    grp = log_win_cand.groupby(["cmdb_id", "log_name"], dropna=False)
    agg = grp.agg(
        incident_point_count=("value", "count"),
        mean_value=("value", "mean"),
        max_value=("value", "max"),
        count_nonzero=("value", lambda s: int((s > 0).sum()))
    ).reset_index()
    # Filter rows where log_name == 'log.error_count' OR count_nonzero > 0
    cond = (agg["log_name"] == "log.error_count") | (agg["count_nonzero"] > 0)
    log_summary = agg[cond].sort_values(["count_nonzero", "incident_point_count"], ascending=[False, False]).head(20).reset_index(drop=True)

# Load error_logs.csv if present
if os.path.exists("error_logs.csv"):
    error_logs = pd.read_csv("error_logs.csv")
    error_logs["timestamp"] = pd.to_datetime(pd.to_numeric(error_logs.get("timestamp"), errors="coerce"), unit="s", utc=True)
    # Filter to incident window and checkoutservice
    err_win = error_logs[(error_logs["timestamp"] >= start) & (error_logs["timestamp"] <= end) & (error_logs["cmdb_id"] == "checkoutservice")].copy()
    # Select up to 20 rows with timestamp and message
    if err_win.empty:
        error_rows = pd.DataFrame(columns=["timestamp", "message"])
        error_count = 0
    else:
        error_rows = err_win[["timestamp", "message"]].sort_values("timestamp").head(20).reset_index(drop=True)
        error_count = int(err_win.shape[0])
else:
    # no file present
    error_rows = pd.DataFrame(columns=["timestamp", "message"])
    error_count = 0

# Display compact results
log_summary, error_rows, error_count

```
Out[7]:
```
Summary for the incident window (2024-01-17 16:26:17 → 16:56:17 UTC):

- Log metrics:
  - For the candidate services, log.row_count had nonzero values for all entries (incident_point_count = 25 for each):
    - checkoutservice: mean row_count 79.80, max 102
    - currencyservice: mean 2262.56, max 2561
    - emailservice: mean 26.60, max 34
    - recommendationservice: mean 532.04, max 585
  - log.error_count is 0 across all candidate services (incident_point_count = 25, mean = 0, max = 0, count_nonzero = 0).

- Error logs:
  - No error_log rows were found for checkoutservice in the incident window (count = 0).

Conclusion: there were no log error spikes or recorded error log messages for checkoutservice during the window. Row counts show normal (nonzero) traffic; any observed service problems (e.g., latency from traces/metrics) are not accompanied by application error logs for these services.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id         log_name  incident_point_count  mean_value  max_value  count_nonzero
0        checkoutservice    log.row_count                    25       79.80        102             25
1        currencyservice    log.row_count                    25     2262.56       2561             25
2           emailservice    log.row_count                    25       26.60         34             25
3  recommendationservice    log.row_count                    25      532.04        585             25
4        checkoutservice  log.error_count                    25        0.00          0              0
5        currencyservice  log.error_count                    25        0.00          0              0
6           emailservice  log.error_count                    25        0.00          0              0
7  recommendationservice  log.error_count                    25        0.00          0              0, Empty DataFrame
Columns: [timestamp, message]
Index: [], 0)```
```